# 03 - SFT LoRA Training: Logs, QA, and Multi-task Adapters

This notebook trains task-specific LoRA adapters starting from the base model.

The following adapters are trained:
- Logs classification adapter;
- QA generation adapter;
- Multi-task adapter.

Only the resulting LoRA adapter weights are saved.
The original base model remains unchanged.

In [ ]:
import exp_common as E

RUN_SEEDS = [42]  # then [123], [777]
MAX_SEQ_LENGTH = 2048
SFT_EPOCHS = 1.0
SFT_MAX_STEPS = -1 
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 2e-4

TASKS = {
    'logs': {
        'train': E.PREPARED_DIR / 'logs_train_sft.jsonl',
        'val': E.PREPARED_DIR / 'logs_val_sft.jsonl',
    },
    'qa': {
        'train': E.PREPARED_DIR / 'qa_train_sft.jsonl',
        'val': E.PREPARED_DIR / 'qa_val_sft.jsonl',
    },
}

MULTITASK_TRAIN = E.PREPARED_DIR / 'multitask_train_sft.jsonl'

In [ ]:
for task_name, paths in TASKS.items():
    train_rows = E.read_jsonl(paths['train'])
    val_rows = E.read_jsonl(paths['val'])
    print(f"\n{task_name}: train={len(train_rows)}, val={len(val_rows)}")
    for seed in RUN_SEEDS:
        out_dir = E.ADAPTERS_DIR / task_name / f'seed_{seed}'
        print('Training', task_name, 'seed', seed)
        E.train_lora_adapter(
            train_rows, val_rows, out_dir, seed=seed, max_seq_length=MAX_SEQ_LENGTH,
            epochs=SFT_EPOCHS, max_steps=SFT_MAX_STEPS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=LR,
            lora_r=16, lora_alpha=32, lora_dropout=0.05,
        )
print('Done')

# Multitask LoRA

In [ ]:
import exp_common as E

RUN_SEEDS = [777]  #[42, 123, 777]
MAX_SEQ_LENGTH = 2048
SFT_EPOCHS = 1.0
SFT_MAX_STEPS = -1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 5e-5 

TASKS = {
    'logs': {
        'train': E.PREPARED_DIR / 'logs_train_sft.jsonl',
        'val': E.PREPARED_DIR / 'logs_val_sft.jsonl',
    },
    'qa': {
        'train': E.PREPARED_DIR / 'qa_train_sft.jsonl',
        'val': E.PREPARED_DIR / 'qa_val_sft.jsonl',
    },
}

MULTITASK_TRAIN = E.PREPARED_DIR / 'multitask_train_sft.jsonl'

In [ ]:
multi_train = E.read_jsonl(MULTITASK_TRAIN)

logs_val = E.read_jsonl(E.PREPARED_DIR / 'logs_val_sft.jsonl')
qa_val = E.read_jsonl(E.PREPARED_DIR / 'qa_val_sft.jsonl')
multi_val = logs_val + qa_val

print(f"Multi-task: train={len(multi_train)}, val={len(multi_val)}")
print(f"  Val contains: Logs={len(logs_val)}, QA={len(qa_val)}")

for seed in RUN_SEEDS:
    out_dir = E.ADAPTERS_DIR / 'multitask' / f'seed_{seed}'
    print('Training multitask, seed', seed)
    E.train_lora_adapter(
        multi_train,
        multi_val,
        out_dir,
        seed=seed,
        max_seq_length=MAX_SEQ_LENGTH,
        epochs=SFT_EPOCHS,
        max_steps=SFT_MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_ratio=0.3,
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.05,
    )
print('Done')